# Business Recommendations

This notebook builds the numbers behind `executive_memo.md`: confidence intervals, a composite ranking of the 3 channels, and a budget allocation table for the $500K monthly budget.

Input: `data/marketing_data.csv`.

Note on consistency: I use the mean of daily CPA/ROAS ratios here (not total cost / total conversions), because that is what `statistical_analysis.ipynb` and `power_analysis.ipynb` actually tested. Using a different average here would not match the significance results I already found.

In [1]:
import numpy as np
import pandas as pd

daily = pd.read_csv('data/marketing_data.csv')
daily.head()

,platform,date,impressions,clicks,conversions,ad_spend,revenue,CTR,conversion_rate,CPA,ROAS
0,Google Ads,2024-01-01,21702,1000,17,3170.00,3670.88,0.046079,0.017000,186.470588,1.158006
1,Google Ads,2024-01-04,162825,7382,452,19953.23,24924.40,0.045337,0.061230,44.144314,1.249141
2,Google Ads,2024-01-05,196692,6264,310,15668.95,64727.90,0.031847,0.049489,50.545000,4.130966
3,Google Ads,2024-01-06,139158,3061,118,3887.47,15835.46,0.021997,0.038549,32.944661,4.073462
4,Google Ads,2024-01-07,238097,9087,340,21959.64,49123.74,0.038165,0.037416,64.587176,2.237001


## Summary of statistically significant findings

From `statistical_analysis.ipynb`, after FDR correction:
- All 3 CPA pairs stayed significant, with small to large effect sizes (Cohen's d from 0.36 up to 1.02).
- All 3 conversion rate pairs stayed significant too, though the actual gaps are under 1 percentage point.

I rebuild the CPA summary below so the numbers are visible in this notebook as well.

In [2]:
cpa_by_channel = daily.groupby('platform')['CPA'].mean().sort_values()
roas_by_channel = daily.groupby('platform')['ROAS'].mean().sort_values(ascending=False)

print('Average CPA by channel (lower is better):')
print(cpa_by_channel.round(2))
print()
print('Average ROAS by channel (higher is better):')
print(roas_by_channel.round(2))

Average CPA by channel (lower is better):
platform
TikTok Ads    27.08
Meta Ads      34.76
Google Ads    57.54
Name: CPA, dtype: float64

Average ROAS by channel (higher is better):
platform
TikTok Ads    9.10
Meta Ads      6.84
Google Ads    3.89
Name: ROAS, dtype: float64


## Confidence intervals for CPA

A single average CPA number hides how much uncertainty is behind it. I use bootstrap resampling to get a 95% confidence interval per channel.

In [3]:
def bootstrap_ci(data, n_bootstrap=1000, ci_level=0.95, rng=None):
    """95% (by default) bootstrap confidence interval for the mean of data."""
    if rng is None:
        rng = np.random.default_rng()

    data = np.asarray(data)
    n = len(data)
    boot_means = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample = rng.choice(data, size=n, replace=True)
        boot_means[i] = sample.mean()

    alpha = 1 - ci_level
    lower = np.percentile(boot_means, 100 * alpha / 2)
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return lower, upper

In [4]:
ci_rows = []
for platform, group in daily.groupby('platform'):
    cpa_values = group['CPA'].dropna().values
    lower, upper = bootstrap_ci(cpa_values)
    ci_rows.append({
        'platform': platform,
        'mean_cpa': cpa_values.mean(),
        'ci_95_lower': lower,
        'ci_95_upper': upper,
    })

ci_df = pd.DataFrame(ci_rows).sort_values('mean_cpa').reset_index(drop=True)
ci_df.round(2)

,platform,mean_cpa,ci_95_lower,ci_95_upper
0,TikTok Ads,27.08,24.75,29.62
1,Meta Ads,34.76,32.40,37.39
2,Google Ads,57.54,53.59,62.05


## Composite ranking and budget allocation

I have two metrics that matter here: CPA (lower is better) and ROAS (higher is better). I rank the channels on each, then combine the ranks 50/50 into one composite score, where a **lower** composite score means a better channel (rank 1 = best on that metric).

In [5]:
metrics = daily.groupby('platform', as_index=False).agg(
    mean_cpa=('CPA', 'mean'),
    mean_roas=('ROAS', 'mean'),
)

metrics['rank_cpa'] = metrics['mean_cpa'].rank(ascending=True)   # rank 1 = lowest CPA = best
metrics['rank_roas'] = metrics['mean_roas'].rank(ascending=False)  # rank 1 = highest ROAS = best
metrics['composite_score'] = 0.5 * metrics['rank_cpa'] + 0.5 * metrics['rank_roas']

metrics = metrics.sort_values('composite_score').reset_index(drop=True)
metrics.round(3)

,platform,mean_cpa,mean_roas,rank_cpa,rank_roas,composite_score
0,TikTok Ads,27.084,9.099,1.0,1.0,1.0
1,Meta Ads,34.764,6.840,2.0,2.0,2.0
2,Google Ads,57.539,3.890,3.0,3.0,3.0


### Turn the ranking into a budget allocation

I convert the composite score into a positive weight (better channel = bigger weight), normalize it to shares of the $500K budget, then apply a floor of 15% and a cap of 50% per channel. The floor keeps me from fully abandoning a channel based on 90 days of data, the cap keeps me from putting all eggs in one basket.

In [6]:
TOTAL_BUDGET = 500_000
MIN_SHARE = 0.15
MAX_SHARE = 0.50

max_composite = metrics['composite_score'].max()
weight = (max_composite - metrics['composite_score']) + 1  # better channel (lower composite) gets more weight
raw_share = weight / weight.sum()

clipped_share = raw_share.clip(lower=MIN_SHARE, upper=MAX_SHARE)
final_share = clipped_share / clipped_share.sum()  # renormalize so shares add up to 1 again

metrics['allocation_pct'] = final_share.values
metrics['allocation_amount'] = (final_share.values * TOTAL_BUDGET).round(0)

allocation_table = metrics[['platform', 'composite_score', 'allocation_pct', 'allocation_amount']]
allocation_table

,platform,composite_score,allocation_pct,allocation_amount
0,TikTok Ads,1.0,0.500000,250000.0
1,Meta Ads,2.0,0.333333,166667.0
2,Google Ads,3.0,0.166667,83333.0


### Compare to current spend

In [7]:
current_spend = daily.groupby('platform')['ad_spend'].sum()
current_share = (current_spend / current_spend.sum()).rename('current_share')

comparison = allocation_table.set_index('platform').join(current_share)
comparison['current_amount_equiv'] = (comparison['current_share'] * TOTAL_BUDGET).round(0)
comparison[['current_share', 'current_amount_equiv', 'allocation_pct', 'allocation_amount']].round(3)

,current_share,current_amount_equiv,allocation_pct,allocation_amount
platform,,,,
TikTok Ads,0.239,119429.0,0.500,250000.0
Meta Ads,0.190,94793.0,0.333,166667.0
Google Ads,0.572,285778.0,0.167,83333.0


### Key insight

Right now, Google Ads gets about 57% of spend even though it has the highest CPA and lowest ROAS of the 3 channels. TikTok Ads, the strongest channel on both metrics, currently gets under 24%. The recommended allocation flips this: TikTok Ads up to 50%, Meta Ads to about 33%, Google Ads down to about 17% (not zero, since 90 days is not forever and Google still brings real volume).

This allocation logic and the exact numbers feed directly into `executive_memo.md`.